# Stage 5b. End-to-End Fine-tuning with Augmentation and Backbone Comparison

Stage 5 menemukan bahwa embedding deep ResNet18-CSA beku (bobot ImageNet, tidak pernah dilatih ulang untuk nail) tidak menambah sinyal di atas fitur hand-crafted, tujuh konfigurasi mengelompok ketat pada AUC 0.70 sampai 0.72 tidak peduli bobot loss atau kapasitas trunk diubah bagaimana pun. Notebook ini menguji hipotesis bahwa embedding-nya sendiri yang generik, bukan arsitektur multi-task-nya, dengan melatih CSA dan proyeksi lewat backpropagation penuh atas citra (badan backbone tetap beku, mengikuti pola yang sama dengan palm dan conjunctiva). Augmentasi ringan (flip, rotasi kecil, brightness/contrast) selalu aktif karena percobaan serupa pada conjunctiva tanpa augmentasi gagal total (MAE memburuk). Dua backbone dibandingkan (ResNet18 vs MobileNetV3-Small) dan bobot kelas severity dicoba terpisah, sama seperti pola stage 5b pada palm, supaya keputusan arsitektur diuji empiris pada data nail sendiri.

## Environment Setup

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent.parent))

import pandas as pd
import torch

from configs import paths
from src.common import manifest as manifest_utils, train

output_dir = paths.outputs_dir("nail")
artifact_dir = paths.artifacts_dir("nail")

manifest = pd.read_csv(output_dir / "manifest.csv")
manifest = manifest[manifest["roi_precropped"]].reset_index(drop=True)
manifest = manifest_utils.assign_kfold(manifest, n_splits=5, seed=42)
handcrafted = pd.read_csv(output_dir / "handcrafted_features.csv")

print("manifest", manifest.shape)
print("device", "cuda" if torch.cuda.is_available() else "cpu")

## Configurations

Empat konfigurasi end-to-end dibandingkan, menyilangkan pilihan backbone dengan aktif tidaknya bobot kelas severity. Augmentasi selalu aktif pada semua konfigurasi di notebook ini karena sudah menjadi prasyarat dasar, bukan variabel yang diuji terpisah.

In [ ]:
configurations = {
    "e2e_resnet18": dict(backbone_name="resnet18", weight_severity_classes=False),
    "e2e_resnet18_weighted_severity": dict(backbone_name="resnet18", weight_severity_classes=True),
    "e2e_mobilenet_v3_small": dict(backbone_name="mobilenet_v3_small", weight_severity_classes=False),
    "e2e_mobilenet_v3_small_weighted_severity": dict(backbone_name="mobilenet_v3_small", weight_severity_classes=True),
}

## Train All Configurations

AUC dan balanced accuracy dilaporkan berdampingan dengan accuracy karena dataset nail timpang (32.6 persen anemic), mengikuti diagnosis pada Stage 5.

In [ ]:
results = {}
for name, overrides in configurations.items():
    print("melatih", name)
    results[name] = train.run_kfold_end_to_end(
        manifest, handcrafted,
        n_splits=5, epochs=40, batch_size=32, learning_rate=1e-3,
        augment=True, **overrides,
    )
    fold_metrics = results[name]["fold_metrics"]
    print(
        name,
        "MAE", round(fold_metrics["mae"].mean(), 4),
        "accuracy", round(fold_metrics["accuracy"].mean(), 4),
        "f1", round(fold_metrics["f1"].mean(), 4),
        "auc", round(fold_metrics["auc"].mean(), 4),
        "balanced_accuracy", round(fold_metrics["balanced_accuracy"].mean(), 4),
        "severity_accuracy", round(fold_metrics["severity_accuracy"].mean(), 4),
    )

## Verify CSA Actually Learned

Memastikan badan backbone tetap beku dan CSA benar-benar menerima gradien selama pelatihan, bukan sekadar klaim, mengikuti pola verifikasi yang sama dengan palm dan conjunctiva stage 5b.

In [ ]:
from src.common import features

sample_model = results["e2e_resnet18"]["models"][0]
backbone = sample_model.embedding_backbone
print("early requires_grad (harus False)", next(backbone.early.parameters()).requires_grad)
print("late requires_grad (harus False)", next(backbone.late.parameters()).requires_grad)
print("csa requires_grad (harus True)", next(backbone.csa.parameters()).requires_grad)
print("projection requires_grad (harus True)", backbone.projection.weight.requires_grad)

fresh_backbone = features.EmbeddingBackbone(backbone_name="resnet18")
csa_weight_diff = (
    backbone.csa.channel_attention.mlp[0].weight.cpu() - fresh_backbone.csa.channel_attention.mlp[0].weight
).abs().mean().item()
print("rerata selisih bobot CSA terlatih vs inisialisasi acak baru (harus jauh dari nol)", round(csa_weight_diff, 6))

## Compare Against Frozen-Embedding Baseline

Dibandingkan jujur dengan hasil Stage 5 (embedding dibekukan, CSA belum terlatih), apa pun hasilnya. AUC dan balanced accuracy jadi pembanding utama, bukan accuracy mentah, karena kelas timpang.

In [ ]:
from sklearn.metrics import cohen_kappa_score

baseline_table = pd.read_csv(output_dir / "multitask_model_comparison.csv")

comparison_rows = list(baseline_table.to_dict(orient="records"))
for name, result in results.items():
    fold_metrics = result["fold_metrics"]
    oof = result["oof"]
    severity_valid = oof["severity_true"] >= 0
    kappa = (
        cohen_kappa_score(oof.loc[severity_valid, "severity_true"], oof.loc[severity_valid, "severity_pred"])
        if severity_valid.any() else float("nan")
    )
    comparison_rows.append({
        "configuration": name,
        "mae": fold_metrics["mae"].mean(),
        "rmse": fold_metrics["rmse"].mean(),
        "accuracy": fold_metrics["accuracy"].mean(),
        "precision": fold_metrics["precision"].mean(),
        "recall": fold_metrics["recall"].mean(),
        "f1": fold_metrics["f1"].mean(),
        "auc": fold_metrics["auc"].mean(),
        "balanced_accuracy": fold_metrics["balanced_accuracy"].mean(),
        "severity_accuracy": fold_metrics["severity_accuracy"].mean(),
        "severity_kappa": kappa,
    })

comparison_table = pd.DataFrame(comparison_rows)
comparison_table.round(4)

## Save Results

Checkpoint per fold disimpan untuk setiap konfigurasi end-to-end agar dapat dipakai ulang pada stage evaluasi ablation tanpa melatih ulang. Konfigurasi terbaik dipilih berdasarkan AUC, bukan MAE atau accuracy mentah, mengikuti diagnosis Stage 5 bahwa AUC adalah metrik yang lebih adil pada dataset timpang ini.

In [ ]:
comparison_table.to_csv(output_dir / "multitask_model_comparison_e2e.csv", index=False)

best_name = max(
    (name for name in results),
    key=lambda name: results[name]["fold_metrics"]["auc"].mean(),
)
results[best_name]["oof"].to_csv(output_dir / "multitask_oof_e2e_best.csv", index=False)
print("konfigurasi end-to-end terbaik berdasarkan AUC", best_name)

for name, result in results.items():
    for fold_index, fold_model in enumerate(result["models"]):
        checkpoint_path = artifact_dir / f"end_to_end_{name}_fold{fold_index}.pt"
        torch.save(fold_model.state_dict(), checkpoint_path)

print("saved end-to-end results and checkpoints to", output_dir, "and", artifact_dir)